# Healthcare Data Solutions – Notebook Deployer

This notebook deploys HDS notebooks from the `dist` folder to the current Fabric workspace with production-ready formatting, using a **modular, phase-based pipeline**.

## High-level workflow
1. **Configuration** – Set artifact version, source `dist` path, environment name, and logical lakehouse names.
2. **Lakehouse Mapping** – Map notebook filenames to logical lakehouse keys.
3. **Helper Functions** – Utilities for workspace/environment resolution, file I/O, and notebook JSON handling.
4. **Formatting Helpers** – Functions that apply production formatting to each notebook.
5. **Deployment Helpers** – Functions that call Fabric REST APIs to create/update notebooks.
6. **Phase Orchestration** – `phase0`–`phase4` functions plus a single `run_deployment()` entry point that executes the full pipeline.
7. **Summary** – A structured `result_summary` plus console output showing successes and failures.

> **Usage**: Update only the configuration cell when you change artifact version, source location, or environment. Then run the final orchestration cell (`run_deployment()`). The rest of the notebook is fully dynamic per workspace.


---

## 1. Configuration Parameters

**Configure your deployment settings below:**

Modify the following variables to match your environment:

### Artifact Configuration:
- `ARTIFACT_VERSION`: Version number for artifacts
- `WORKSPACE_NAME`: OneLake container hosting the dist folder
- `ARTIFACT_LAKEHOUSE_NAME`: Lakehouse name containing artifacts

### OneLake Connection:
- `ONELAKE_ENDPOINT`: Auto-detected from Fabric or uses default

### Deployment Configuration:
- `TARGET_ENVIRONMENT_NAME`: Environment name to attach to notebooks (auto-detect if empty)
- `SOLUTION_NAME`: Healthcare solution display name
- `ADMIN_DB_NAME`: Administration database logical name
- `LOGICAL_LAKEHOUSES`: Mapping of logical lakehouse names

### Deployment Options:
- `SAVE_FORMATTED_LOCALLY`: Save formatted notebooks to lakehouse
- `DEPLOY_TO_WORKSPACE`: Deploy notebooks to Fabric workspace

> **Note:** These variables feed into all subsequent phases (scan, format, deploy) and are the **only** values you typically need to edit.


In [ ]:
%run common_deployment_config


In [ ]:
# Notebook-specific imports for notebook deployment
import os
import uuid
import traceback

print("✓ Notebook-specific imports loaded")

In [ ]:
# ============================================================================
# NOTEBOOK-SPECIFIC CONFIGURATION
# ============================================================================
# (Common config loaded from common_deployment_config)

# Construct notebook-specific path from base
print("✓ Notebook deployer configuration:")
print(f"  Workspace: {WORKSPACE_NAME}")
print(f"  Version: {ARTIFACT_VERSION}")
print(f"  Environment: {TARGET_ENVIRONMENT_NAME}")
print(f"  Lakehouse count: {len(LOGICAL_LAKEHOUSES)}")

## 2. Lakehouse Mapping

**Note:** Lakehouse mappings (NOTEBOOK_LAKEHOUSE_MAPPING, EXCLUDED_FROM_MARKDOWN_REMOVAL, EXCLUDED_FROM_LAKEHOUSE_DEPENDENCY) are loaded from the centralized configuration in `common_deployment_config.ipynb`.

The mapping is used by the **formatting phase** to attach the correct lakehouse dependency metadata to each deployed notebook.

## 3. Helper Functions

Utility helpers for:
- Workspace resolution (`get_current_workspace_id`)
- Environment resolution (`get_environment_id`)
- Reading and writing notebook JSON definitions (`read_notebook_file`, `write_notebook_file`)
- Safe file operations in OneLake (`mssparkutils.fs`)

These helpers are reused across the subsequent formatting and deployment phases.

In [ ]:
def create_notebook_config(notebook_path: str, notebook_name: str, capability: str,
                          default_lakehouse: Optional[str] = None,
                          default_lakehouse_id: Optional[str] = None,
                          workspace_id: Optional[str] = None,
                          environment_id: Optional[str] = None) -> Dict[str, Any]:
    """
    Create a notebook configuration dictionary.
    
    Args:
        notebook_path: Path to the notebook file
        notebook_name: Name of the notebook
        capability: Capability name
        default_lakehouse: Logical lakehouse name (optional)
        default_lakehouse_id: Lakehouse ID (optional)
        workspace_id: Workspace ID (optional)
        environment_id: Environment ID (optional)
        
    Returns:
        Dictionary containing notebook configuration
    """
    return {
        "notebook_path": notebook_path,
        "notebook_name": notebook_name,
        "capability": capability,
        "default_lakehouse": default_lakehouse,
        "default_lakehouse_id": default_lakehouse_id,
        "workspace_id": workspace_id,
        "environment_id": environment_id
    }



def get_current_workspace_id():
    """
    Get current workspace ID.
    
    Returns:
        Workspace ID string
        
    Raises:
        RuntimeError: If unable to resolve workspace ID
    """
    try:
        return fabric.resolve_workspace_id()
    except:
        return mssparkutils.env.getWorkspaceId()


def read_notebook_file(path: str) -> Dict[str, Any]:
    """Read and parse notebook JSON file"""
    content = mssparkutils.fs.head(path, 50000000)
    return json.loads(content)


def write_notebook_file(path: str, data: Dict[str, Any]) -> None:
    """Write notebook JSON to file"""
    content = json.dumps(data, indent=2)
    temp_path = f"/tmp/{uuid.uuid4()}.ipynb"
    
    try:
        # Write to temp file
        with open(temp_path, "w") as f:
            f.write(content)
        
        # Copy to destination
        mssparkutils.fs.cp(f"file://{temp_path}", path, True)
    finally:
        # Always cleanup temp file
        if os.path.exists(temp_path):
            os.remove(temp_path)


def get_environment_id(environment_name: str = None, workspace_id: str = None) -> tuple:
    """
    Get environment ID from name with smart matching.
    
    Args:
        environment_name: Environment name to match (optional)
        workspace_id: Workspace ID (defaults to current workspace)
        
    Returns:
        Tuple of (environment_id, environment_name)
    """
    ws_id = workspace_id or WORKSPACE_ID
    env_name = environment_name
    
    try:
        response = FABRIC_CLIENT.get(f"/v1/workspaces/{ws_id}/environments")
        if response.status_code != 200:
            print(f"⚠ Could not list environments (status {response.status_code})")
            return None, None
        
        data = json.loads(response.text)
        environments = data.get("value", [])
        
        if not environments:
            print("⚠ No environments found in workspace")
            return None, None
        
        # If only 1 environment, use it
        if len(environments) == 1:
            env = environments[0]
            env_id = env.get("id")
            env_display = env.get("displayName")
            print(f"  ✓ Auto-selected only environment: '{env_display}'")
            return env_id, env_display
        
        # Multiple environments - match by name
        if not env_name:
            print(f"  Found {len(environments)} environments. Specify environment name:")
            for env in environments:
                print(f"    • {env.get('displayName')}")
            return None, None
        
        search_name = env_name.lower().strip()
        
        # Exact match
        for env in environments:
            if env.get("displayName", "").lower().strip() == search_name:
                env_id = env.get("id")
                env_display = env.get("displayName")
                print(f"  ✓ Environment matched (exact): '{env_display}'")
                return env_id, env_display
        
        # Partial match
        for env in environments:
            if search_name in env.get("displayName", "").lower().strip():
                env_id = env.get("id")
                env_display = env.get("displayName")
                print(f"  ✓ Environment matched (partial): '{env_display}'")
                return env_id, env_display
        
        print(f"  ✗ No environment matching '{env_name}' found")
        return None, None
        
    except Exception as e:
        print(f"✗ Error getting environment: {e}")
        return None, None

## 4. Scanning the `dist` Folder

`scan_dist_folder` walks the `healthcare-artifacts/<version>` tree to discover capabilities and their `Notebooks` subfolders.

Each discovered notebook is represented as a dictionary carrying:
- Source path
- Notebook name
- Capability name
- Resolved lakehouse & environment context

These notebook dictionaries are the primary input to the formatting and deployment phases.

In [ ]:
def scan_dist_folder(source_path: str, version: str, workspace_id: str, environment_id: str, lakehouse_ids: Dict[str, str]) -> List[Dict[str, Any]]:
    """
    Scan dist folder for notebooks across all capabilities.
    
    Args:
        source_path: Path to dist folder
        version: Version folder name
        workspace_id: Target workspace ID
        environment_id: Target environment ID
        lakehouse_ids: Dictionary of resolved lakehouse IDs
        
    Returns:
        List of notebook configuration dictionaries
        
    Raises:
        FileNotFoundError: If healthcare-artifacts path does not exist
        RuntimeError: If scanning encounters critical errors
    """
    notebooks = []
    
    # Construct path to healthcare-artifacts
    healthcare_path = f"{source_path}/healthcare-artifacts/{version}"
    
    print(f"📂 Scanning: {healthcare_path}")
    
    # Validate path exists
    if not mssparkutils.fs.exists(healthcare_path):
        error_msg = f"Healthcare artifacts path does not exist: {healthcare_path}"
        print(f"✗ ERROR: {error_msg}")
        raise FileNotFoundError(error_msg)
    
    try:
        # List all capability directories
        capabilities = mssparkutils.fs.ls(healthcare_path)
        
        for cap_item in capabilities:
            if not cap_item.isDir:
                continue
                
            capability = cap_item.name
            notebooks_folder = f"{cap_item.path}/Notebooks"
            
            # If Notebooks folder doesn't exist, skip this capability gracefully
            try:
                if not mssparkutils.fs.exists(notebooks_folder):
                    print(f"  Warning: 'Notebooks' folder not found for capability '{capability}', skipping.")
                    continue
            except Exception as e:
                # If existence check itself fails, log and skip
                print(f"  Warning: Could not check 'Notebooks' folder for {capability}: {e}")
                continue

            # At this point, the Notebooks folder exists – try to list .ipynb files
            try:
                notebook_files = [
                    f for f in mssparkutils.fs.ls(notebooks_folder)
                    if f.name.endswith(".ipynb")
                ]
                
                for nb_file in notebook_files:
                    notebook_name = nb_file.name
                    
                    # Get default lakehouse from mapping
                    lakehouse_name = NOTEBOOK_LAKEHOUSE_MAPPING.get(notebook_name)
                    lakehouse_id = lakehouse_ids.get(lakehouse_name) if lakehouse_name else None
                    
                    config = create_notebook_config(
                        notebook_path=nb_file.path,
                        notebook_name=notebook_name,
                        capability=capability,
                        default_lakehouse=lakehouse_name,
                        default_lakehouse_id=lakehouse_id,
                        workspace_id=workspace_id,
                        environment_id=environment_id
                    )
                    notebooks.append(config)
                    
            except Exception as e:
                print(f"  Warning: No notebooks in {capability}: {e}")
                
    except Exception as e:
        print(f"Error scanning dist folder: {e}")
        
    return notebooks

## 5. Notebook Formatting Helpers

These functions encapsulate production formatting rules applied in **Phase 1**:
- Add lakehouse and environment dependencies
- Adjust `%run` commands where needed
- Optionally remove markdown cells
- Insert a warning header
- Remove `pip install` cells
- Lock code cells to prevent edits

The orchestration function `format_notebook_for_production` composes these helpers into a single step used by the phase pipeline.

In [ ]:
def add_percent_signs_from_run_command(notebook_data: Dict[str, Any]) -> None:
    """Add %% around notebook names in %run commands"""
    for cell in notebook_data.get("cells", []):
        if cell.get("cell_type") == "code" and "source" in cell:
            for idx, source in enumerate(cell["source"]):
                if source.startswith("%run"):
                    # Remove any existing %% first
                    modified_source = source.replace("%%", "")
                    # Add %% around the config notebook name
                    modified_source = modified_source.replace(
                        "msft_dm4h_setup_and_config_notebook",
                        "%%msft_dm4h_setup_and_config_notebook%%"
                    )
                    cell["source"][idx] = modified_source


def remove_pip_install(notebook_data: Dict[str, Any]) -> Dict[str, Any]:
    """Remove cells containing pip install commands"""
    filtered_cells = []
    for cell in notebook_data.get("cells", []):
        remove_cell = False
        if cell.get("cell_type") == "code" and "source" in cell:
            for source in cell["source"]:
                if "pip install" in source:
                    remove_cell = True
                    break
        if not remove_cell:
            filtered_cells.append(cell)
    notebook_data["cells"] = filtered_cells
    return notebook_data


# def lock_cells(notebook_data: Dict[str, Any]) -> None:
#     """Lock all code cells"""
#     for cell in notebook_data.get("cells", []):
#         if cell.get("cell_type") == "code":
#             if "metadata" not in cell:
#                 cell["metadata"] = {}
#             cell["metadata"]["run_control"] = {"frozen": False}
#             cell["metadata"]["editable"] = False


def add_markdown_header(notebook_data: Dict[str, Any]) -> None:
    """Add warning header to notebook"""
    header = {
        "cell_type": "markdown",
        "id": str(uuid.uuid4()),
        "metadata": {},
        "source": [
            "##### WARNING\n",
            "The following notebook is intended to be read only. Please do not modify the contents of this notebook.\n"
        ]
    }
    notebook_data["cells"].insert(0, header)


def remove_markdown_cells(notebook_data: Dict[str, Any]) -> None:
    """Remove all markdown cells"""
    filtered_cells = [cell for cell in notebook_data.get("cells", [])
                     if cell.get("cell_type") == "code"]
    notebook_data["cells"] = filtered_cells


def remove_trident_section_and_add_dependencies(notebook_data: Dict[str, Any], notebook_name: str) -> None:
    """Convert trident section to dependencies"""
    if "metadata" in notebook_data:
        if "trident" in notebook_data["metadata"]:
            notebook_data["metadata"]["dependencies"] = notebook_data["metadata"]["trident"]
            del notebook_data["metadata"]["trident"]
        if "dependencies" not in notebook_data["metadata"]:
            notebook_data["metadata"]["dependencies"] = {}


def add_lakehouse_dependency_node(notebook_data: Dict[str, Any], notebook_name: str,
                                   lakehouse_id: str, lakehouse_name: str, workspace_id: str) -> None:
    """Add default lakehouse dependency"""
    if "metadata" not in notebook_data:
        notebook_data["metadata"] = {}
    
    default_lakehouse_blob = {
        "default_lakehouse": lakehouse_id,
        "default_lakehouse_name": lakehouse_name,
        "default_lakehouse_workspace_id": workspace_id,
    }
    
    remove_trident_section_and_add_dependencies(notebook_data, notebook_name)
    notebook_data["metadata"]["dependencies"]["lakehouse"] = default_lakehouse_blob


def add_environment_dependency_node(notebook_data: Dict[str, Any], notebook_name: str,
                                    environment_id: str, workspace_id: str) -> None:
    """Add environment dependency"""
    if "metadata" not in notebook_data:
        notebook_data["metadata"] = {}
    
    default_environment_blob = {
        "environmentId": environment_id,
        "workspaceId": workspace_id,
    }
    
    remove_trident_section_and_add_dependencies(notebook_data, notebook_name)
    notebook_data["metadata"]["dependencies"]["environment"] = default_environment_blob

In [ ]:
def format_notebook_for_production(notebook_data: Dict[str, Any], 
                                   notebook_name: str,
                                   lakehouse_id: Optional[str],
                                   lakehouse_name: Optional[str],
                                   workspace_id: str,
                                   environment_id: str) -> Dict[str, Any]:
    """
    Apply production-level formatting to notebook
    """
    # 1. Add lakehouse dependency (if not excluded)
    if (notebook_name not in EXCLUDED_FROM_LAKEHOUSE_DEPENDENCY and lakehouse_id is not None):
        add_lakehouse_dependency_node(notebook_data, notebook_name, lakehouse_id, 
                                     lakehouse_name, workspace_id)
        print(f"    ✓ Lakehouse '{lakehouse_name}' added")
    
    # 2. Add environment dependency
    add_environment_dependency_node(notebook_data, notebook_name, environment_id, workspace_id)
    print(f"    ✓ Environment dependency added")
    
    # 3. Add percent signs to %run commands
    add_percent_signs_from_run_command(notebook_data)
    
    # 4. Remove markdown cells (if not excluded)
    if notebook_name not in EXCLUDED_FROM_MARKDOWN_REMOVAL:
        remove_markdown_cells(notebook_data)
        print(f"    ✓ Markdown cells removed")
    
    # 5. Add warning header
    add_markdown_header(notebook_data)
    
    # 6. Remove pip install cells
    notebook_data = remove_pip_install(notebook_data)
    
    # # 7. Lock cells
    # lock_cells(notebook_data)
    # print(f"    ✓ Cells locked")
    
    return notebook_data

## 6. Deployment Helpers (Fabric REST APIs)

These functions wrap the Fabric REST APIs (via `FabricRestClient`) to:
- List existing notebooks in the workspace
- Create notebooks if they don’t exist
- Update notebooks if they already exist

They operate on the formatted notebook JSON (serialized as `ipynb`, base64-encoded) and are invoked from **Phase 3** of the orchestration pipeline.

In [ ]:
def list_notebooks(client: FabricRestClient, workspace_id: str) -> List[Dict]:
    """List all notebooks in the workspace"""
    try:
        response = client.get(f"/v1/workspaces/{workspace_id}/notebooks")
        if response.status_code == 200:
            data = json.loads(response.text)
            return data.get("value", [])
        return []
    except Exception as e:
        print(f"    Warning: Could not list notebooks: {e}")
        return []


def _build_ipynb_part(notebook_data: Dict[str, Any]) -> Dict[str, Any]:
    """Build a valid ipynb part payload for Fabric item APIs."""
    # Serialize ipynb JSON and base64-encode it
    raw_json = json.dumps(notebook_data)
    b64_payload = base64.b64encode(raw_json.encode("utf-8")).decode("utf-8")

    return {
        "path": "notebook.ipynb",          # path name is arbitrary but should be .ipynb
        "payload": b64_payload,             # base64 of the ipynb content
        "payloadType": "InlineBase64",     # content is base64-encoded
    }


def create_notebook(client: FabricRestClient, workspace_id: str, display_name: str, notebook_data: Dict[str, Any]) -> bool:
    """Create a new notebook in the workspace"""
    try:
        part = _build_ipynb_part(notebook_data)

        payload = {
            "displayName": display_name,
            "definition": {
                "format": "ipynb",
                "parts": [part],
            },
        }

        response = client.post(f"/v1/workspaces/{workspace_id}/notebooks", json=payload)
        if response.status_code not in [200, 201, 202]:
            print(f"    ✗ Create failed, status: {response.status_code}")
            try:
                print(f"    Error: {response.text[:200]}")
            except Exception:
                pass
        return response.status_code in [200, 201, 202]

    except Exception as e:
        print(f"    Error creating notebook: {e}")
        return False


def update_notebook(client: FabricRestClient, workspace_id: str, notebook_id: str,
                   notebook_data: Dict[str, Any]) -> bool:
    """Update an existing notebook"""
    try:
        part = _build_ipynb_part(notebook_data)

        payload = {
            "definition": {
                "format": "ipynb",
                "parts": [part],
            },
        }

        response = client.post(
            f"/v1/workspaces/{workspace_id}/notebooks/{notebook_id}/updateDefinition",
            json=payload,
        )
        if response.status_code not in [200, 202]:
            print(f"    ✗ Update failed, status: {response.status_code}")
            try:
                print(f"    Error: {response.text[:200]}")
            except Exception:
                pass
        return response.status_code in [200, 202]

    except Exception as e:
        print(f"    Error updating notebook: {e}")
        return False


def deploy_notebook_to_fabric(client: FabricRestClient, config: Dict[str, Any],
                              notebook_data: Dict[str, Any]) -> bool:
    """Deploy notebook to Fabric workspace using FabricRestClient with prefix handling"""
    # Build display name with smart prefix handling (avoids duplicating technical prefix)
    notebook_display_name = build_notebook_display_name(config["notebook_name"])
    
    try:
        # Check if notebook already exists
        existing_notebooks = list_notebooks(client, config["workspace_id"])
        existing_notebook = next(
            (nb for nb in existing_notebooks if nb.get("displayName") == notebook_display_name),
            None,
        )
        
        if existing_notebook:
            # Update existing notebook
            notebook_id = existing_notebook["id"]
            print(f"    Updating existing notebook: {notebook_display_name}")
            success = update_notebook(client, config["workspace_id"], notebook_id, notebook_data)
        else:
            # Create new notebook
            print(f"    Creating new notebook: {notebook_display_name}")
            success = create_notebook(client, config["workspace_id"], notebook_display_name, notebook_data)
        
        if success:
            print(f"    ✓ Successfully deployed as '{notebook_display_name}'")
            return True
        else:
            print(f"    ✗ Deployment failed")
            return False
            
    except Exception as e:
        print(f"    ✗ Error: {str(e)}")
        return False

## 7. Main Deployment Orchestration

The main deployment orchestration is implemented through a `main()` function that coordinates all phases:

**Phase 0 – Initialization (`phase0_init`)**
- Resolve current workspace ID
- Resolve lakehouse IDs
- Create `FabricRestClient`
- Resolve target environment

**Scanning – Discover Notebooks**
- Scan dist folder for notebooks
- Create configuration dictionaries for each notebook
- Group notebooks by capability

**Phase 1 – Format & Save Temp (`phase1_format_and_save_temp`)**
- Read each source notebook
- Apply production formatting
- Save formatted copies to a temporary folder

**Phase 2 – Update Config Placeholders (`phase2_update_config_placeholders`)**
- Perform a second pass over the config notebook
- Replace any remaining placeholders (workspace, lakehouse IDs, solution ID)
- Optionally report unresolved placeholders

**Phase 3 – Deployment (`phase3_deploy`)**
- Deploy formatted notebooks (create or skip if already present)
- Track successes and failures by capability

**Phase 4 – Cleanup (`phase4_cleanup`)**
- Remove the temporary formatted notebooks folder

The `main()` function returns a structured result summary with deployment statistics.


In [ ]:
def replace_notebooks_placeholders(notebook_data: Dict[str, Any],
                                workspace_id: str,
                                lakehouse_ids: Dict[str, str],
                                solution_name: str,
                                TECHNICAL_PREFIX: str = "msft") -> Dict[str, Any]:
    """Replace placeholders in notebook cells using IDs.

    Rules:
      - %%workspace_id%% / %%workspace_name%% -> current workspace_id
      - %%<logical>_id%% -> that lakehouse's ID (e.g., %%msft_bronze_id%%)
      - %%administration_database_name%% -> msft_admin lakehouse ID (if available)
      - %%solution_name%% -> SOLUTION_ID (if available, with smart prefix matching)
    """
    replacements: Dict[str, str] = {}

    # Workspace-related placeholders
    replacements["%%workspace_id%%"] = str(workspace_id)
    replacements["%%workspace_name%%"] = str(workspace_id)

    # Lakehouse ID placeholders, pattern: %%<logical>_id%%
    for logical_name, lh_id in lakehouse_ids.items():
        if not lh_id:
            continue
        placeholder = f"%%{logical_name}_id%%"
        replacements[placeholder] = str(lh_id)

    # Lakehouse database_name placeholders, pattern: %%<base_name>_database_name%%
    # This handles placeholders like %%customer_insights_database_name%%, %%poa_gold_database_name%%, etc.
    # Strategy: Extract base name from logical key, then map back to ID
    database_name_pattern = re.compile(r"%%([a-zA-Z0-9_]+)_database_name%%")
    
    # Build a reverse mapping: base_name -> lakehouse_id
    base_name_to_id = {}
    for logical_name, lh_id in lakehouse_ids.items():
        if not lh_id:
            continue
        # Extract base name by removing technical prefix if present
        if TECHNICAL_PREFIX and logical_name.startswith(f"{TECHNICAL_PREFIX}_"):
            base_name = logical_name[len(TECHNICAL_PREFIX)+1:]
        else:
            base_name = logical_name
        base_name_to_id[base_name] = lh_id
    
    # Also support lookup with technical prefix (for explicit naming)
    for logical_name, lh_id in lakehouse_ids.items():
        if lh_id:
            base_name_to_id[logical_name] = lh_id
    
    # Pre-scan all cells to find database_name placeholders and resolve them
    for cell in notebook_data.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src_lines = cell.get("source", []) or []
        for line in src_lines:
            for match in database_name_pattern.finditer(line):
                base_name = match.group(1)  # e.g., "customer_insights", "poa_gold"
                placeholder = match.group(0)  # e.g., "%%customer_insights_database_name%%"
                
                if placeholder in replacements:
                    continue  # Already resolved
                
                # Try direct lookup first
                lh_id = base_name_to_id.get(base_name)
                if lh_id:
                    replacements[placeholder] = str(lh_id)
                    print(f"  Resolved {placeholder} -> {lh_id} (via base name '{base_name}')")

    # administration_database_name -> admin lakehouse id (from ADMIN_DB_NAME config)
    # Use dynamic lookup based on TECHNICAL_PREFIX instead of hardcoded "msft_admin"
    admin_logical_name = f"{TECHNICAL_PREFIX}_admin" if TECHNICAL_PREFIX else "admin"
    admin_id = lakehouse_ids.get(admin_logical_name)
    if admin_id:
        replacements["%%administration_database_name%%"] = str(admin_id)
    else:
        # Fallback: search for any lakehouse with base name "admin"
        for logical_name, lh_id in lakehouse_ids.items():
            if logical_name.endswith("_admin") or logical_name == "admin":
                admin_id = lh_id
                replacements["%%administration_database_name%%"] = str(lh_id)
                break

    # solution_name -> SOLUTION_ID, with smart admin lakehouse detection
    # Strategy 0: If solution_name matches admin DB base name and admin_id exists, use it directly
    # This handles the common case where SOLUTION_NAME = ADMIN_DB_NAME = "admin"
    sol_id = None
    if solution_name and admin_id:
        # Check if solution_name is "admin" (the typical case)
        if solution_name == "admin":
            sol_id = admin_id
            print(f"SOLUTION_NAME '{solution_name}' resolved to admin lakehouse id {sol_id}")
    
    # Strategy 1: Direct lookup (in case solution_name is already a logical key like "msft_admin")
    if not sol_id:
        sol_id = lakehouse_ids.get(solution_name)
        if sol_id:
            print(f"SOLUTION_NAME '{solution_name}' resolved via direct lookup to id {sol_id}")
    
    # Strategy 2: Try with technical prefix (e.g., "admin" -> "msft_admin")
    if not sol_id and solution_name and TECHNICAL_PREFIX:
        prefixed_solution_name = f"{TECHNICAL_PREFIX}_{solution_name}"
        sol_id = lakehouse_ids.get(prefixed_solution_name)
        if sol_id:
            print(f"SOLUTION_NAME '{solution_name}' resolved via prefixed lookup '{prefixed_solution_name}' to id {sol_id}")
    
    # Strategy 3: Try matching against lakehouse display names (base names without prefix)
    if not sol_id and solution_name:
        # Search for a lakehouse whose base name matches solution_name
        for logical_name, lh_id in lakehouse_ids.items():
            # Extract base name by removing technical prefix if present
            if TECHNICAL_PREFIX and logical_name.startswith(f"{TECHNICAL_PREFIX}_"):
                base_name = logical_name[len(TECHNICAL_PREFIX)+1:]
            else:
                base_name = logical_name
            
            if base_name == solution_name and lh_id:
                sol_id = lh_id
                print(f"SOLUTION_NAME '{solution_name}' resolved via base name match in '{logical_name}' to id {sol_id}")
                break
    
    if sol_id:
        replacements["%%solution_name%%"] = str(sol_id)
    else:
        print(f"WARNING: Could not resolve SOLUTION_NAME '{solution_name}' to an id.")
        print(f"  Available lakehouse keys: {list(lakehouse_ids.keys())}")

    # Apply replacements to code cell sources
    for cell in notebook_data.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src_lines = cell.get("source", []) or []

        new_lines = []
        for line in src_lines:
            for ph, value in replacements.items():
                if ph in line:
                    line = line.replace(ph, value)
            new_lines.append(line)
        cell["source"] = new_lines

    return notebook_data

In [ ]:
def phase0_init() -> tuple[str, FabricRestClient, str, str, Dict[str, str]]:
    """Initialize workspace, client, environment, and lakehouse mappings.

    Returns: (workspace_id, client, environment_id, resolved_environment_name, lakehouse_ids)
    """
    print("=" * 80)
    print("HEALTHCARE DATA SOLUTIONS - NOTEBOOK DEPLOYER")
    print("=" * 80)

    # Use workspace ID from common config (already auto-detected)
    print("\nGetting workspace information...")
    workspace_id = WORKSPACE_ID
    print(f"✓ Workspace ID: {workspace_id}")

    # Resolve lakehouse IDs
    print("\nResolving lakehouses...")
    lakehouse_ids = resolve_lakehouse_ids(LAKEHOUSE_NAME_MAP)

    # Initialize Fabric client
    client = FabricRestClient()
    print("✓ FabricRestClient initialized")

    # Get environment ID from name
    print("\nResolving environment...")
    # Build prefixed environment name to match what was created
    prefixed_env_name = build_artifact_name(TARGET_ENVIRONMENT_NAME)
    environment_id, resolved_env_name = get_environment_id(
        prefixed_env_name,
        workspace_id,
    )

    if not environment_id:
        print("\n⚠ ERROR: Could not resolve environment ID")
        print("Please check TARGET_ENVIRONMENT_NAME configuration")
        raise ValueError("Environment ID required for deployment")

    print(f"✓ Environment ID: {environment_id}")
    return workspace_id, client, environment_id, resolved_env_name, lakehouse_ids


def phase1_format_and_save_temp(
    notebooks: List[Dict[str, Any]],
    workspace_id: str,
    environment_id: str,
    lakehouse_ids: Dict[str, str],
    solution_name: str,
    TECHNICAL_PREFIX: str = "msft") -> tuple[Dict[str, str], Optional[str], Dict[str, str], str]:
    """Phase 1: format notebooks and save to temporary folder.

    Returns:
        temp_paths: notebook_name -> temp formatted path
        config_temp_path: path of formatted msft_config_notebook (if any)
        formatting_failure_reasons: notebook_name -> error string
        temp_formatted_root: root folder used for formatted notebooks
    """
    print("\n" + "=" * 80)
    print("PHASE 1: FORMAT & SAVE TO TEMP")
    print("=" * 80)

    temp_formatted_root = f"{BASE_DIST_PATH}/_formatted_tmp_{uuid.uuid4()}"
    print(f"Temporary formatted notebooks path: {temp_formatted_root}")

    if SAVE_FORMATTED_LOCALLY:
        try:
            mssparkutils.fs.mkdirs(temp_formatted_root)
        except Exception:
            # Folder may already exist; ignore
            pass

    temp_paths: Dict[str, str] = {}
    config_temp_path: Optional[str] = None
    formatting_failure_reasons: Dict[str, str] = {}

    for idx, config in enumerate(notebooks, 1):
        print(f"\n[{idx}/{len(notebooks)}] {config['notebook_name']}")
        print(f"  Capability: {config['capability']}")
        print(f"  Lakehouse: {config['default_lakehouse'] or 'None'}")

        try:
            # Read notebook from source
            print("  Reading notebook...")
            notebook_data = read_notebook_file(config["notebook_path"])

            # Replace placeholders in ALL notebooks (not just config notebook)
            print("  Replacing placeholders...")
            notebook_data = replace_notebooks_placeholders(
                notebook_data,
                workspace_id=workspace_id,
                lakehouse_ids=lakehouse_ids,
                solution_name=solution_name,
                TECHNICAL_PREFIX=TECHNICAL_PREFIX,
            )

            # Apply production formatting
            print("  Formatting for production...")
            formatted_data = format_notebook_for_production(
                notebook_data,
                config["notebook_name"],
                config["default_lakehouse_id"],
                config["default_lakehouse"],
                config["workspace_id"],
                config["environment_id"],
            )

            # Save formatted version to a temporary folder (not source location)
            if SAVE_FORMATTED_LOCALLY:
                temp_capability_folder = f"{temp_formatted_root}/{config['capability']}"
                try:
                    mssparkutils.fs.mkdirs(temp_capability_folder)
                except Exception:
                    # Folder may already exist; ignore
                    pass

                formatted_path = f"{temp_capability_folder}/{config['notebook_name']}"
                write_notebook_file(formatted_path, formatted_data)
                temp_paths[config["notebook_name"]] = formatted_path

                if config["notebook_name"] == "msft_config_notebook.ipynb":
                    config_temp_path = formatted_path

                print("    ✓ Saved formatted version to temp path")
            else:
                temp_paths[config["notebook_name"]] = config["notebook_path"]

        except Exception as e:
            reason = str(e)
            print(f"  ✗ Error during formatting/saving: {reason}")
            traceback.print_exc()
            formatting_failure_reasons[config["notebook_name"]] = reason

    return temp_paths, config_temp_path, formatting_failure_reasons, temp_formatted_root


def phase2_update_config_placeholders(
    temp_paths: Dict[str, str],
    workspace_id: str,
    lakehouse_ids: Dict[str, str],
    solution_name: str,
    TECHNICAL_PREFIX: str = "msft") -> Dict[str, List[str]]:
    """Phase 2: Scan all notebooks for unresolved placeholders.

    Returns:
        unresolved_placeholders_by_notebook: notebook_name -> list of placeholders/errors
    """
    print("\n" + "=" * 80)
    print("PHASE 2: VERIFY PLACEHOLDER RESOLUTION IN ALL NOTEBOOKS")
    print("=" * 80)

    unresolved_placeholders_by_notebook: Dict[str, List[str]] = {}

    if not temp_paths:
        print("No notebooks to check. Skipping placeholder verification phase.")
        return unresolved_placeholders_by_notebook

    # Scan for any remaining %%...%% patterns in code cells
    pattern = re.compile(r"%%[^%]+%%")

    for notebook_name, temp_path in temp_paths.items():
        try:
            print(f"\nChecking: {notebook_name}")
            nb_data = read_notebook_file(temp_path)

            unresolved_placeholders: set[str] = set()
            for cell in nb_data.get("cells", []):
                if cell.get("cell_type") != "code":
                    continue
                for line in cell.get("source", []) or []:
                    for match in pattern.findall(line):
                        unresolved_placeholders.add(match)

            if unresolved_placeholders:
                sorted_ph = sorted(unresolved_placeholders)
                unresolved_placeholders_by_notebook[notebook_name] = sorted_ph
                print(f"  ⚠ Found {len(sorted_ph)} unresolved placeholder(s):")
                for ph in sorted_ph:
                    print(f"    - {ph}")
            else:
                print(f"  ✓ All placeholders resolved")

        except Exception as e:
            print(f"  ✗ Error checking placeholders: {e}")
            unresolved_placeholders_by_notebook[notebook_name] = [f"ERROR: {e}"]

    # Summary
    if unresolved_placeholders_by_notebook:
        print(f"\n⚠ WARNING: {len(unresolved_placeholders_by_notebook)} notebook(s) have unresolved placeholders")
    else:
        print("\n✓ All notebooks have placeholders successfully resolved")

    return unresolved_placeholders_by_notebook


def phase3_deploy(
    client: FabricRestClient,
    notebooks: List[Dict[str, Any]],
    workspace_id: str,
    temp_paths: Dict[str, str],
    formatting_failure_reasons: Dict[str, str]) -> tuple[Dict[str, Dict[str, List[str]]], Dict[str, str], int, int, int]:
    """Phase 3: deploy formatted notebooks to workspace.

    Returns:
        deployment_summary, failure_reasons, success_count, failure_count, skip_count

        deployment_summary has three categories (by capability):
          - "deployed": newly created or updated notebooks
          - "failed":   notebooks that failed deployment
          - "skipped":  notebooks that already existed and were skipped
    """
    print("\n" + "=" * 80)
    print("PHASE 3: DEPLOYMENT")
    print("=" * 80)

    deployment_summary: Dict[str, Dict[str, List[str]]] = {"deployed": {}, "failed": {}, "skipped": {}}
    failure_reasons: Dict[str, str] = {}
    success_count = 0
    failure_count = 0
    skip_count = 0

    existing_notebooks_in_ws = list_notebooks(client, workspace_id)
    existing_notebook_names = {nb.get("displayName") for nb in existing_notebooks_in_ws}

    for idx, config in enumerate(notebooks, 1):
        print(f"\n[{idx}/{len(notebooks)}] {config['notebook_name']}")
        print(f"  Capability: {config['capability']}")
        print(f"  Lakehouse: {config['default_lakehouse'] or 'None'}")

        # Build display name with smart prefix handling
        notebook_display_name = build_notebook_display_name(config["notebook_name"])

        try:
            formatted_path = temp_paths.get(config["notebook_name"])
            if not formatted_path:
                reason = "no formatted temp path found"
                if config["notebook_name"] in formatting_failure_reasons:
                    reason = (
                        "formatting failed in Phase 1: "
                        f"{formatting_failure_reasons[config['notebook_name']]}"
                    )
                print(f"  ✗ {reason}, skipping.")
                failure_count += 1
                deployment_summary["failed"].setdefault(config["capability"], []).append(
                    config["notebook_name"],
                )
                failure_reasons[config["notebook_name"]] = reason
                continue

            notebook_data = read_notebook_file(formatted_path)

            if notebook_display_name in existing_notebook_names:
                print(f"  Notebook already exists in workspace as '{notebook_display_name}', skipping deployment.")
                deployment_summary["skipped"].setdefault(config["capability"], []).append(
                    config["notebook_name"],
                )
                skip_count += 1
                continue

            print(f"  Deploying to workspace as '{notebook_display_name}'...")
            if deploy_notebook_to_fabric(client, config, notebook_data):
                success_count += 1
                deployment_summary["deployed"].setdefault(config["capability"], []).append(
                    config["notebook_name"],
                )
                existing_notebook_names.add(notebook_display_name)
            else:
                reason = "deployment function returned failure"
                failure_count += 1
                deployment_summary["failed"].setdefault(config["capability"], []).append(
                    config["notebook_name"],
                )
                failure_reasons[config["notebook_name"]] = reason

        except Exception as e:
            reason = str(e)
            print(f"  ✗ Error during deployment: {reason}")
            traceback.print_exc()
            failure_count += 1
            deployment_summary["failed"].setdefault(config["capability"], []).append(
                config["notebook_name"],
            )
            failure_reasons[config["notebook_name"]] = reason

    return deployment_summary, failure_reasons, success_count, failure_count, skip_count


def phase4_cleanup(temp_formatted_root: str) -> None:
    """Phase 4: clean up temporary formatted notebooks folder."""
    if not SAVE_FORMATTED_LOCALLY:
        return

    try:
        mssparkutils.fs.rm(temp_formatted_root, recurse=True)
        print(f"\n✓ Temporary formatted notebooks folder deleted: {temp_formatted_root}")
    except Exception as e:
        print(f"\n⚠ Could not delete temp folder {temp_formatted_root}: {e}")

In [ ]:
def start_notebook_deployment() -> Dict[str, Any]:
    """
    Main deployment orchestration function.
    
    Executes all phases:
    - Phase 0: Initialize workspace, client, environment, lakehouse mappings
    - Scanning: Discover notebooks in dist folder
    - Phase 1: Format notebooks and save to temp
    - Phase 2: Update config placeholders
    - Phase 3: Deploy to workspace
    - Phase 4: Cleanup temp folder
    
    Returns:
        Summary dictionary with deployment results
    """
    try:
        # Phase 0 – Initialize
        workspace_id, client, environment_id, resolved_env_name, lakehouse_ids = phase0_init()

        # Scanning
        print("\n" + "=" * 80)
        print("SCANNING FOR NOTEBOOKS")
        print("=" * 80)

        notebooks = scan_dist_folder(
            BASE_DIST_PATH,
            ARTIFACT_VERSION,
            workspace_id,
            environment_id,
            lakehouse_ids,
        )

        print(f"\n✓ Found {len(notebooks)} notebooks")
        if not notebooks:
            error_msg = "No notebooks found. Check BASE_DIST_PATH and ARTIFACT_VERSION."
            print(f"\n✗ ERROR: {error_msg}")
            raise FileNotFoundError(error_msg)

        # Group by capability (info only)
        by_capability: Dict[str, List[str]] = {}
        for nb in notebooks:
            by_capability.setdefault(nb["capability"], []).append(nb["notebook_name"])

        print("\nNotebooks by capability:")
        for cap, nbs in sorted(by_capability.items()):
            print(f"  • {cap}: {len(nbs)} notebook(s)")

        # Phase 1 – Format and save to temp
        (temp_paths, config_temp_path, formatting_failure_reasons, temp_formatted_root) = phase1_format_and_save_temp(
            notebooks, workspace_id, environment_id, lakehouse_ids, SOLUTION_NAME, TECHNICAL_PREFIX)

        # Phase 2 – Verify placeholders in all notebooks
        unresolved_placeholders_by_notebook = phase2_update_config_placeholders(
            temp_paths, workspace_id, lakehouse_ids, SOLUTION_NAME, TECHNICAL_PREFIX)

        # Phase 3 – Deploy to workspace
        (deployment_summary, failure_reasons, success_count, failure_count, skip_count) = phase3_deploy(client, notebooks,workspace_id, temp_paths, formatting_failure_reasons)

        # ATTENTION: AI-generated code can include errors or operations you didn't intend. Review the code in this cell carefully before running it.

        # Phase 4 – Cleanup
        phase4_cleanup(temp_formatted_root)

        # Compute counts
        total = len(notebooks)
        deployed_count = success_count
        skipped_count = sum(len(nbs) for nbs in deployment_summary.get("skipped", {}).values())

        success_rate = (deployed_count / total * 100) if total else 0.0
        skip_rate = (skipped_count / total * 100) if total else 0.0

        # Print summary
        print("\n" + "=" * 80)
        print("DEPLOYMENT SUMMARY")
        print("=" * 80)
        print(f"Total notebooks: {total}")
        print(f"Deployed (new/updated): {deployed_count} ({success_rate:.1f}%)")
        print(f"Skipped (already existed): {skipped_count} ({skip_rate:.1f}%)")
        print(f"Failed: {failure_count}")
        print(f"\nWorkspace ID: {workspace_id}")
        print(f"Environment: {resolved_env_name} ({environment_id})")
        print(f"Artifact Version: {ARTIFACT_VERSION}")
        print("=" * 80)

        print("\nDeployed notebooks by capability:")
        if deployment_summary["deployed"]:
            for cap, nbs in sorted(deployment_summary["deployed"].items()):
                print(f"  • {cap}:")
                for nb in sorted(nbs):
                    print(f"      - {nb}")
        else:
            print("  (none)")

        print("\nSkipped notebooks by capability:")
        if deployment_summary.get("skipped"):
            for cap, nbs in sorted(deployment_summary["skipped"].items()):
                print(f"  • {cap}:")
                for nb in sorted(nbs):
                    print(f"      - {nb}")
        else:
            print("  (none)")

        print("\nFailed notebooks by capability:")
        if deployment_summary["failed"]:
            for cap, nbs in sorted(deployment_summary["failed"].items()):
                print(f"  • {cap}:")
                for nb in sorted(nbs):
                    print(f"      - {nb}")
        else:
            print("  (none)")

        if failure_reasons:
            print("\nFailure reasons per notebook:")
            for nb_name, reason in sorted(failure_reasons.items()):
                print(f"  - {nb_name}: {reason}")

        if failure_count == 0 and DEPLOY_TO_WORKSPACE:
            print("\n✓ All notebooks deployed successfully (new or already present).")
        elif not DEPLOY_TO_WORKSPACE:
            print("\n⊘ Deployment was skipped (DEPLOY_TO_WORKSPACE=False).")
        else:
            print(f"\n⚠ {failure_count} notebook(s) failed. Check errors above.")

        return {
            "unresolved_placeholders_by_notebook": unresolved_placeholders_by_notebook,
            "deployment_summary": deployment_summary,
            "failure_reasons": failure_reasons,
            "formatting_failure_reasons": formatting_failure_reasons,
            "success_count": deployed_count,
            "failure_count": failure_count,
            "skipped_count": skipped_count,
            "success_rate": success_rate,
            "skip_rate": skip_rate,
        }
    except Exception as e:
        # Phase 4 – Cleanup
        phase4_cleanup(temp_formatted_root)
        print(f"\n✗ Unexpected error in main: {e}")

In [ ]:
# Execute deployment
start_notebook_deployment()